# Hello

## 1. Load Data

In [ ]:
from pathlib import Path
DATA_DOWNLOAD_DIR = Path("/Volumes") / "Crucial X9" # replace with your dataset path

In [ ]:
# Dataset Metadata
import pandas as pd

metadata_df = pd.read_csv(DATA_DOWNLOAD_DIR / "emg2pose_metadata.csv")
metadata_df.head(5)

In [ ]:
# Display users
import glob
import os

users = sorted([
    p for p in Path(DATA_DOWNLOAD_DIR, "emg_dataset/by_user").iterdir()
    if p.is_dir()
])

for i, x in enumerate(users):
    print(f"{i}: {x}")

In [ ]:
# Display sessions for chosen user

user = users[0] # choose a user

sessions = sorted(glob.glob(os.path.join(user, "*.hdf5")))
for i, x in enumerate(sessions):
    print(f"{i}: {x}")

## 2. Select Session To Evaluate On

In [ ]:
from emg2pose.data import Emg2PoseSessionData

# Select our dataset 
session = sessions[0]
print(session)
data = Emg2PoseSessionData(hdf5_path=session)

In [ ]:
# View the data
print(data.fields)
print(f"{'emg shape: ':<20} {data['emg'].shape}")
print(f"{'joint_angles shape: ':<20} {data['joint_angles'].shape}")
print(f"{'time shape: ':<20} {data['time'].shape}")

In [ ]:
# Stream EMG
from experiments.stream_emg import EmgStreamer
from IPython.display import display, update_display
disp = display("", display_id=True)

stream = EmgStreamer(data)

size = data['emg'].shape[0]

# for i in range(size):
#     disp.update(str(stream.step()))

In [ ]:
from emg2pose.utils import downsample

joint_angles_30hz = downsample(data['emg'], native_fs=2000, target_fs=30)
joint_angles_30hz.shape

In [ ]:
mask = data.no_ik_failure
mask

In [ ]:
# Show our sessions metadata

metadata_df[metadata_df["filename"] == data.metadata["filename"]]

In [ ]:
# Downscale ground-truth joint angles and display it over time

import emg2pose.visualization as visualization
from emg2pose.utils import downsample
import numpy as np

joint_angles = data["joint_angles"]
joint_angles_30hz = downsample(joint_angles, native_fs=2000, target_fs=30)

visualization.get_plotly_animation_for_joint_angles(joint_angles_30hz[0:250])

In [ ]:
# Show where inverse kinematics tracking (ground truth labeling)
visualization.ik_failure_plot(data)

## 3. Generate Predictions Of This Session
Load the EMG data in to the models

### Select Session(s) to Train On
Certain methods require training our own model using various ML methods.
We use our defined `sessions` list that are avaialbe to train on
and set start and end indeces to decide what subset of `sessions` to train on.

In [ ]:
DATA_REGIMES = ["single_session", "single_user", "multi_user", "full"]

from experiments.load_data import load_data

user_train_dict = load_data(DATA_REGIMES[0], [users[0]])
user_train_dict

In [ ]:
from experiments.trainers import _concat_sessions

train = _concat_sessions(user_train_dict)
print(train)

### 3a. Load Checkpoint and Predict
Loads a large pretrained model from Meta Reality labs and runs it on the data

In [ ]:
from pathlib import Path
import os

checkpoint_dir = Path(DATA_DOWNLOAD_DIR) / "emg2pose_model_checkpoints"

# Download checkpoint if it does not exist
if not checkpoint_dir.exists():
    os.system(f'''
    cd {DATA_DOWNLOAD_DIR} &&
    curl "https://fb-ctrl-oss.s3.amazonaws.com/emg2pose/emg2pose_model_checkpoints.tar.gz" -o emg2pose_model_checkpoints.tar.gz &&
    tar -xvzf emg2pose_model_checkpoints.tar.gz
    ''')

In [ ]:
from emg2pose.utils import generate_hydra_config_from_overrides

config = generate_hydra_config_from_overrides(
    overrides=[
        "experiment=tracking_vemg2pose",
        f"checkpoint={DATA_DOWNLOAD_DIR}/emg2pose_model_checkpoints/regression_vemg2pose.ckpt"
    ]
)

In [ ]:
from emg2pose.lightning import Emg2PoseModule

module = Emg2PoseModule.load_from_checkpoint(
    config.checkpoint,
    network=config.network,
    optimizer=config.optimizer,
    lr_scheduler=config.lr_scheduler,
)

In [ ]:
# all data
import torch

from experiments.inference import emg2pose_inferece

preds, joint_angles, no_ik_failure = emg2pose_inferece(data, module)

print("Predictions Shape: " + str(preds.shape))
print("Ground Truth Shape " + str(joint_angles.shape))

In [ ]:
# Visualize the predictions

preds_30hz = downsample(preds, native_fs=2000, target_fs=30)
visualization.get_plotly_animation_for_joint_angles(preds_30hz[0:250], color="gray")

In [ ]:
# Ensure alignment and scale

print(preds.shape)
print(joint_angles.shape)

preds_30hz = downsample(preds, native_fs=2000, target_fs=30)
ground_truth_30hz = downsample(joint_angles, native_fs=2000, target_fs=30)

print(preds_30hz.shape)
print(ground_truth_30hz.shape)

mask = no_ik_failure.cpu().numpy().squeeze()  # -> (T,)
mask_30hz = downsample(mask.astype(float), 2000, 30) > 0.5
mask_30hz = mask_30hz.squeeze()  # ensure (T,)

preds_30hz = preds_30hz[mask_30hz]
ground_truth_30hz = ground_truth_30hz[mask_30hz]

print(preds_30hz.shape)
print(ground_truth_30hz.shape)


In [ ]:
from experiments.evaluation import evaluate_predictions

evaluate_predictions(preds_30hz, ground_truth_30hz)

In [ ]:
import importlib
import experiments.inference as str

importlib.reload(str)

In [ ]:
# Stream EMG
from experiments.stream_emg import EmgStreamer
from IPython.display import display, update_display, clear_output
from experiments.inference import emg2pose_window_inference
disp = display("", display_id=True)
import time

stream = EmgStreamer(data)
size = data['emg'].shape[0]

buf = []

WINDOW = 12000
STRIDE = 50
WARMUP = 10

latency_buf = []

for i in range(size):

    emg_t = stream.step()
    buf.append(emg_t)

    # keep buffer fixed size
    if len(buf) > WINDOW:
        buf.pop(0)

    # only run when buffer full AND at interval
    if len(buf) == WINDOW and i % STRIDE == 0:
        window = np.stack(buf, axis=0)  # (T, C)

        t0 = time.perf_counter()

        preds = emg2pose_window_inference(window, module)

        t1 = time.perf_counter()

        clear_output(wait=True)

        print(f"Predictions Shape {i}: {preds.shape}")
        
        latency_ms = (t1 - t0) * 1000

        latency_buf.append(latency_ms)

        print(f"Inference time: {latency_ms:.2f} ms")

        mean = np.mean(latency_buf[WARMUP:])
        median = np.median(latency_buf[WARMUP:])

        print(f"Mean inference time: {mean:.2f} ms")
        print(f"Median inference time: {median:.2f} ms")

        # fig = visualization.plot_hand_mesh(preds[-1], auto_range=False)
        # display(fig)

        

In [ ]:
from experiments.stream_emg import stream_inference

mean_latency, median_latency, _ = stream_inference(data, emg2pose_window_inference, module, WINDOW=12000)
print(mean_latency, median_latency)

### 3b Small LSTM

In [ ]:
from experiments.trainers import train_small_lstm
from experiments.load_data import concat_data

emg, joint_angles_lstm = concat_data(user_train_dict)
small_lstm_model, seq_len, ds_factor, stride = train_small_lstm(emg, joint_angles_lstm)

In [ ]:
import importlib
import emg2pose.feature_extraction as fe

importlib.reload(fe)

In [ ]:
import importlib
import experiments.inference as r

importlib.reload(r)

In [ ]:
from experiments.stream_emg import stream_inference
from experiments.inference import lstm_window_inference

mean_latency, median_latency, _ = stream_inference(data, lstm_window_inference, small_lstm_model, WINDOW=seq_len, ds_factor=ds_factor)
print(f"mean latency {mean_latency:.2f} ms, median latency {median_latency:.2f} ms")

In [ ]:
from experiments.inference import small_lstm_inference
from experiments.metrics import get_experiment_metrics

preds, y_gt, mask_lstm = small_lstm_inference(
        data, small_lstm_model, seq_len, ds_factor, stride
    )
print("\n[Small LSTM]")
print(get_experiment_metrics(preds, y_gt, mask_lstm))

In [ ]:
effective_hz = 500 / stride

gt_30hz = downsample(y_gt, effective_hz, 30)
preds_30hz = downsample(preds, effective_hz, 30)

print(gt_30hz.shape)
print(preds_30hz.shape)

In [ ]:
visualization.get_plotly_animation_for_joint_angles(preds_30hz[0:250], color="gray")

In [ ]:
visualization.get_plotly_animation_for_joint_angles(ground_truth_30hz[0:250], color="pink")

### 3c Meta Model with Limited Training Data

In [ ]:
import importlib
import experiments.trainers as tr

importlib.reload(tr)

In [ ]:
from experiments.trainers import train_emg2pose
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = ""

my_emg2pose_model = train_emg2pose(user_train_dict, DATA_DOWNLOAD_DIR, epochs=2)

In [ ]:
# My EMG2Pose 
preds, joint_angles, no_ik_failure = emg2pose_inferece(data, my_emg2pose_model)
print("\n[My EMG2Pose]")
print(get_experiment_metrics(preds, joint_angles, no_ik_failure))

In [ ]:
# Visualize the predictions

preds_30hz = downsample(preds, native_fs=2000, target_fs=30)
visualization.get_plotly_animation_for_joint_angles(preds_30hz[0:250], color="gray")

In [ ]:
# Visualize the gt

ground_truth_30hz = downsample(joint_angles, native_fs=2000, target_fs=30)
visualization.get_plotly_animation_for_joint_angles(joint_angles_30hz[0:250], color="pink")

In [ ]:
preds_30hz = downsample(preds, native_fs=2000, target_fs=30)
ground_truth_30hz = downsample(joint_angles, native_fs=2000, target_fs=30)

mask = no_ik_failure.cpu().numpy().squeeze()
mask_30hz = downsample(mask.astype(float), 2000, 30) > 0.5
mask_30hz = mask_30hz.squeeze()

min_len = min(len(preds_30hz), len(ground_truth_30hz), len(mask_30hz))
preds_30hz = preds_30hz[:min_len]
ground_truth_30hz = ground_truth_30hz[:min_len]
mask_30hz = mask_30hz[:min_len]

# THEN APPLY MASK
preds_30hz = preds_30hz[mask_30hz]
ground_truth_30hz = ground_truth_30hz[mask_30hz]

### 3d Classical Machine Learning

In [ ]:
from experiments.trainers import build_features, train_classic_ml

emg_features, joint_angles_ml = build_features(user_train_dict)
ridge_model, svr_model, pls_model = train_classic_ml(emg_features, joint_angles_ml)

In [ ]:
from experiments.inference import classic_ml_inference

ridge_pred, svr_pred, pls_pred, gt, classic_ml_mask = classic_ml_inference(
            data, ridge_model, svr_model, pls_model
        )
print("\n[Ridge]")
print(get_experiment_metrics(ridge_pred, gt, classic_ml_mask))
print("\n[SVR]")
print(get_experiment_metrics(svr_pred, gt, classic_ml_mask))
print("\n[PLS]")
print(get_experiment_metrics(pls_pred, gt, classic_ml_mask)) 

In [ ]:
from experiments.stream_emg import stream_inference
from experiments.inference import ridge_window_inference, svr_window_inference, pls_window_inference

methods = ["ridge", "svr", "pls"]
functions = [ridge_window_inference, svr_window_inference, pls_window_inference]
models = [ridge_model, svr_model, pls_model]

for i in range(len(methods)):
    mean_latency, median_latency, _ = stream_inference(data, functions[i], models[i], WINDOW=500, STRIDE=67)
    print(f"{methods[i]} | mean latency {mean_latency:.2f} ms, median latency {median_latency:.2f} ms")

In [ ]:
# Ground Truth Visualization

visualization.get_plotly_animation_for_joint_angles(
    gt[0:250],
    color="pink"
)

In [ ]:
# Ridge Pred Visualization

visualization.get_plotly_animation_for_joint_angles(
    ridge_pred[0:250],
    color="gray"
)

In [ ]:
evaluate_predictions(ridge_pred, gt)

In [ ]:
# SVR Pred Visualization

visualization.get_plotly_animation_for_joint_angles(
    svr_pred[0:250],
    color="gray"
)

In [ ]:
evaluate_predictions(svr_pred, gt)

In [ ]:
visualization.get_plotly_animation_for_joint_angles(
    pls_pred[0:250],
    color="gray"
)

In [ ]:
evaluate_predictions(pls_pred, gt)